# Experimento 5 - AION fine-tuning parcial

## Deteccion de lentes gravitacionales fuertes - INF659 Deep Learning (PUCP)

Este notebook parte directamente del Experimento 4 de Renzo, pero reemplaza el flujo de embeddings precalculados por fine-tuning parcial de `polymathic-ai/aion-base`.

Flujo principal:

```text
H5 batch -> preprocesamiento/augmentation -> bandas g,r,i -> LegacySurveyImage
-> CodecManager.encode -> AION.encode entrenable en ultimos bloques
-> mean pooling -> LayerNorm + MLP -> logit binario
```

La pregunta central es si descongelar solo las ultimas capas de AION mejora lo suficiente frente a AION congelado + MLP como para justificar el mayor costo computacional.


## Conceptos clave

**Que se reutiliza de Renzo:** secciones 1-3, carga de AION, mapeo g,r,i a `DES-G/R/I`, `NUM_ENCODER_TOKENS=600`, metricas y protocolo de evaluacion.

**Que cambia:** AION ya no es solo extractor fijo. Se congelan todos sus parametros y luego se descongelan solo los ultimos 1-2 bloques del encoder. Por eso no se usan embeddings precalculados ni `StandardScaler` sobre embeddings dinamicos; la cabeza usa `LayerNorm`.

**Anti-leakage:** normalizacion solo con train, augmentation solo en train, checkpoint y umbral solo con validation, test una sola vez al final.

**TPR0/TPR10:** TPR0 recupera lentes antes del primer falso positivo; TPR10 usa maximo 9 falsos positivos, es decir, menos de 10 FP.


## 1. Entorno de ejecucion

**QUE hace:** prepara librerias, semillas y GPU/Colab.

**POR QUE se hace:** todos los experimentos deben partir de `SEED=42` y ser reproducibles.

**QUE se reutiliza de Renzo:** instalacion condicional de AION y semillas `random`, `numpy`, `torch`, CUDA.

**QUE cambia respecto al Experimento 4:** se usa `EXP5_PRUEBA_RAPIDA` y se agregan utilidades para fine-tuning, scheduler y checkpoints.


In [ ]:
# En Google Colab hay que instalar el paquete oficial de AION en cada sesion.
import importlib.util
import sys

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB and importlib.util.find_spec("aion") is None:
    %pip install --quiet --upgrade polymathic-aion huggingface_hub safetensors


In [ ]:
import json
import math
import os
import random
import re
import shutil
import time
from contextlib import nullcontext
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, log_loss, precision_recall_curve,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedKFold

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PRUEBA_RAPIDA = os.environ.get(
    "EXP5_PRUEBA_RAPIDA",
    "0" if torch.cuda.is_available() else "1",
) == "1"

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)
print("Modo prueba rapida:", PRUEBA_RAPIDA)


### Rutas y opciones

**QUE hace:** localiza `bologna_synthetic_v2.h5` y crea `artifacts_experimento5/`.

**POR QUE se hace:** evita rutas absolutas de una sola computadora.

**QUE se reutiliza de Renzo:** estructura Colab/Drive y copia opcional al runtime.

**QUE cambia:** salida nueva para no mezclar resultados con Experimento 4.


In [ ]:
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DIR = Path("/content/drive/MyDrive/LensDetector15")
    SOURCE_DATASET_PATH = PROJECT_DIR / "data" / "bologna_synthetic_v2.h5"
    ARTIFACTS_DIR = PROJECT_DIR / "artifacts_experimento5"
    COPY_DATASET_TO_RUNTIME = True
    RUNTIME_DATASET_PATH = Path("/content/data/bologna_synthetic_v2.h5")
else:
    PROJECT_DIR = Path.cwd()
    dataset_candidates = [
        PROJECT_DIR / "bologna_synthetic_v2.h5",
        PROJECT_DIR / "data" / "bologna_synthetic_v2.h5",
        PROJECT_DIR / "LensDetector15-20260704T193831Z-3-001" / "bologna_synthetic_v2.h5",
        PROJECT_DIR / "LensDetector15-20260704T193831Z-3-001" / "bologna_synthetic_v2-002.h5",
    ]
    SOURCE_DATASET_PATH = next((p for p in dataset_candidates if p.is_file()), dataset_candidates[0])
    ARTIFACTS_DIR = PROJECT_DIR / "artifacts_experimento5"
    COPY_DATASET_TO_RUNTIME = False
    RUNTIME_DATASET_PATH = SOURCE_DATASET_PATH

if not SOURCE_DATASET_PATH.is_file():
    raise FileNotFoundError(
        f"No se encontro el dataset en {SOURCE_DATASET_PATH}. "
        "Coloca bologna_synthetic_v2.h5 en la raiz del proyecto o en data/."
    )

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if COPY_DATASET_TO_RUNTIME:
    RUNTIME_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
    source_size = SOURCE_DATASET_PATH.stat().st_size
    copy_required = (not RUNTIME_DATASET_PATH.exists() or RUNTIME_DATASET_PATH.stat().st_size != source_size)
    if copy_required:
        print("Copiando el HDF5 al almacenamiento temporal de Colab...")
        shutil.copy2(SOURCE_DATASET_PATH, RUNTIME_DATASET_PATH)
    DATASET_PATH = RUNTIME_DATASET_PATH
else:
    DATASET_PATH = SOURCE_DATASET_PATH

print("Dataset en uso:", DATASET_PATH)
print("Tamano: {:.2f} GB".format(DATASET_PATH.stat().st_size / 1024**3))
print("Resultados:", ARTIFACTS_DIR)


## 2. Datos y particiones

**QUE hace:** valida el HDF5 y respeta `split` (0=train, 1=validation, 2=test).

**POR QUE se hace:** todos los experimentos deben ser comparables y no se debe recalcular un split propio.

**QUE se reutiliza de Renzo:** `h5py`, checks de forma `101x101x4`, balance de clases y semillas.

**QUE cambia:** variables fisicas auxiliares se usan si existen, sin volverlas obligatorias.


In [ ]:
BANDS = ("u", "g", "r", "i")
SPLIT_NAMES = {0: "train", 1: "validation", 2: "test"}
REQUIRED_CORE_DATASETS = {"image", "mask", "is_lens", "split", "sample_seed"}
OPTIONAL_DATASETS = {"theta_E", "source_to_lens_ratio_r", "lensed_snr_r", "negative_type"}

with h5py.File(DATASET_PATH, "r") as h5:
    missing = REQUIRED_CORE_DATASETS.difference(h5.keys())
    if missing:
        raise KeyError(f"Faltan campos obligatorios en el HDF5: {sorted(missing)}")
    available_optional = sorted(OPTIONAL_DATASETS.intersection(h5.keys()))
    print("Campos auxiliares disponibles:", available_optional)

    n_samples = h5["image"].shape[0]
    assert h5["image"].shape == (n_samples, 101, 101, 4)
    assert h5["mask"].shape == (n_samples, 101, 101)
    assert h5["is_lens"].shape == (n_samples,)
    assert h5["split"].shape == (n_samples,)

    labels = h5["is_lens"][:].astype(np.int8)
    split_codes = h5["split"][:].astype(np.int8)
    sample_seeds = h5["sample_seed"][:]

    if "bands" in h5.attrs:
        stored_bands = tuple(json.loads(h5.attrs["bands"]))
        assert stored_bands == BANDS, f"Orden de bandas inesperado: {stored_bands}"

if set(np.unique(labels)) != {0, 1}:
    raise ValueError(f"Etiquetas inesperadas: {np.unique(labels)}")
if set(np.unique(split_codes)) != {0, 1, 2}:
    raise ValueError(f"Particiones inesperadas: {np.unique(split_codes)}")

partition_indices = {name: np.flatnonzero(split_codes == code) for code, name in SPLIT_NAMES.items()}
train_indices = partition_indices["train"]
val_indices = partition_indices["validation"]
test_indices = partition_indices["test"]

for name, indices in partition_indices.items():
    counts = np.bincount(labels[indices], minlength=2)
    print(f"{name:>10}: {len(indices):>6} imagenes | no lente={counts[0]:>5} | lente={counts[1]:>5} | positivos={labels[indices].mean():.3f}")

assert sum(map(len, partition_indices.values())) == n_samples
assert len(np.unique(np.concatenate(list(partition_indices.values())))) == n_samples
print("Semillas repetidas:", n_samples - len(np.unique(sample_seeds)))
print("Confirmacion: test no se usa en entrenamiento ni seleccion de umbral.")


### Subconjunto de prueba rapida

Submuestra estratificada para validar el pipeline cuando no hay GPU o `PRUEBA_RAPIDA=True`.


In [ ]:
def submuestra_estratificada(indices, etiquetas, n_total, semilla):
    rng = np.random.default_rng(semilla)
    elegidos = []
    for clase in (0, 1):
        candidatos = indices[etiquetas[indices] == clase]
        elegidos.append(rng.choice(candidatos, size=min(n_total // 2, len(candidatos)), replace=False))
    return np.sort(np.concatenate(elegidos))

if PRUEBA_RAPIDA:
    indices_uso_train = submuestra_estratificada(train_indices, labels, 160, SEED)
    indices_uso_val = submuestra_estratificada(val_indices, labels, 48, SEED)
    indices_uso_test = submuestra_estratificada(test_indices, labels, 48, SEED)
    print("MODO PRUEBA:", len(indices_uso_train), len(indices_uso_val), len(indices_uso_test))
else:
    indices_uso_train = train_indices
    indices_uso_val = val_indices
    indices_uso_test = test_indices
    print("Corrida completa:", len(indices_uso_train), len(indices_uso_val), len(indices_uso_test))


### Lectura por lotes

Lectura bajo demanda con indices ordenados para `h5py`, igual que Renzo.


In [ ]:
def read_h5_batch(dataset_path, indices):
    # Convertimos a arreglo de enteros de 64 bits (lo que h5py espera)
    indices = np.asarray(indices, dtype=np.int64)
    if indices.ndim != 1 or len(indices) == 0:
        raise ValueError("indices debe ser un vector no vacío")
    if len(np.unique(indices)) != len(indices):
        raise ValueError("El lote contiene índices repetidos")

    order = np.argsort(indices)          # h5py necesita índices crecientes
    sorted_indices = indices[order]
    restore_order = np.argsort(order)    # para devolver el orden pedido

    with h5py.File(dataset_path, "r") as h5:
        return {
            "image": h5["image"][sorted_indices][restore_order],
            "mask": h5["mask"][sorted_indices][restore_order].astype(bool),
            "label": h5["is_lens"][sorted_indices][restore_order].astype(np.int8),
        }


### Inspeccion visual

Composicion RGB i,r,g para detectar problemas obvios antes de usar GPU.


In [ ]:
def rgb_for_display(image):
    # Apilamos i, r, g como canales R, G, B y estiramos el contraste
    rgb = np.stack([image[..., 3], image[..., 2], image[..., 1]], axis=-1)
    low = np.percentile(rgb, 1.0, axis=(0, 1), keepdims=True)
    high = np.percentile(rgb, 99.5, axis=(0, 1), keepdims=True)
    rgb = np.clip((rgb - low) / np.maximum(high - low, 1e-8), 0.0, 1.0)
    return np.sqrt(rgb)   # raíz cuadrada = realce suave de zonas débiles


positive_examples = indices_uso_train[labels[indices_uso_train] == 1][:4]
negative_examples = indices_uso_train[labels[indices_uso_train] == 0][:4]
preview_indices = np.concatenate([positive_examples, negative_examples])
preview = read_h5_batch(DATASET_PATH, preview_indices)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, sample_id, image, label in zip(
    axes.ravel(), preview_indices, preview["image"], preview["label"]
):
    axis.imshow(rgb_for_display(image), origin="lower")
    axis.set_title(f"ID {sample_id} · clase {label}")
    axis.axis("off")
fig.suptitle("Muestra del conjunto de entrenamiento", fontsize=14)
fig.tight_layout()
plt.show()


## 3. Preprocesamiento y aumento

**QUE hace:** percentiles 0.5-99.5 solo con train, estandarizacion, mascara a cero y augmentation solo en train.

**POR QUE se hace:** conserva el pipeline comun y evita data leakage.

**QUE se reutiliza de Renzo:** `preprocess_images`, `augment_batch`, `H5LensDataset` y estadisticas train-only.

**QUE cambia:** `BATCH_SIZE` es pequeno porque ahora AION participa en forward/backward.


In [ ]:
BATCH_SIZE = int(os.environ.get("EXP5_BATCH_SIZE", "4" if DEVICE.type == "cuda" else "2"))
CALIBRATION_IMAGES = 512
LOW_PERCENTILE = 0.5
HIGH_PERCENTILE = 99.5
MAX_TRANSLATION_PIXELS = 3

PREPROCESSING_STATS_PATH = ARTIFACTS_DIR / "preprocessing_stats.json"
dataset_signature = {
    "size_bytes": int(SOURCE_DATASET_PATH.stat().st_size),
    "n_samples": int(n_samples),
    "calibration_images": int(min(CALIBRATION_IMAGES, len(train_indices))),
    "seed": SEED,
}

reuse_stats = False
if PREPROCESSING_STATS_PATH.exists():
    saved_stats = json.loads(PREPROCESSING_STATS_PATH.read_text(encoding="utf-8"))
    reuse_stats = saved_stats.get("dataset_signature") == dataset_signature

if reuse_stats:
    preprocessing_stats = saved_stats
    print("Se reutilizaron las estadisticas guardadas.")
else:
    rng = np.random.default_rng(SEED)
    calibration_indices = np.sort(rng.choice(train_indices, size=min(CALIBRATION_IMAGES, len(train_indices)), replace=False))
    calibration = read_h5_batch(DATASET_PATH, calibration_indices)
    images = calibration["image"].astype(np.float32, copy=False)
    masks = calibration["mask"]
    lower_bounds = np.empty(4, dtype=np.float32)
    upper_bounds = np.empty(4, dtype=np.float32)
    means = np.empty(4, dtype=np.float32)
    stds = np.empty(4, dtype=np.float32)
    for band_index, band in enumerate(BANDS):
        valid = images[..., band_index][~masks]
        lower, upper = np.percentile(valid, [LOW_PERCENTILE, HIGH_PERCENTILE])
        clipped = np.clip(valid, lower, upper)
        lower_bounds[band_index] = lower
        upper_bounds[band_index] = upper
        means[band_index] = clipped.mean(dtype=np.float64)
        stds[band_index] = clipped.std(dtype=np.float64)
        if stds[band_index] <= 0:
            raise ValueError(f"Desviacion invalida en la banda {band}")
    preprocessing_stats = {
        "dataset_signature": dataset_signature, "bands": list(BANDS),
        "low_percentile": LOW_PERCENTILE, "high_percentile": HIGH_PERCENTILE,
        "lower_bounds": lower_bounds.tolist(), "upper_bounds": upper_bounds.tolist(),
        "means": means.tolist(), "stds": stds.tolist(),
    }
    PREPROCESSING_STATS_PATH.write_text(json.dumps(preprocessing_stats, indent=2), encoding="utf-8")
    print("Estadisticas calculadas y guardadas.")

for index, band in enumerate(BANDS):
    print(f"{band}: clip=[{preprocessing_stats['lower_bounds'][index]:.4f}, {preprocessing_stats['upper_bounds'][index]:.4f}] | media={preprocessing_stats['means'][index]:.4f} | std={preprocessing_stats['stds'][index]:.4f}")
print("BATCH_SIZE:", BATCH_SIZE)


In [ ]:
# --- Funciones IDÉNTICAS al piloto (NumPy puro) ---

def preprocess_images(images, masks, stats):
    """Clip a percentiles → estandarizar → píxeles enmascarados a 0."""
    images = np.asarray(images, dtype=np.float32)
    masks = np.asarray(masks, dtype=bool)

    # Los vectores por banda se reacomodan a (1,1,1,4) para difundir sobre HxW
    lower = np.asarray(stats["lower_bounds"], dtype=np.float32).reshape(1, 1, 1, 4)
    upper = np.asarray(stats["upper_bounds"], dtype=np.float32).reshape(1, 1, 1, 4)
    means = np.asarray(stats["means"], dtype=np.float32).reshape(1, 1, 1, 4)
    stds = np.asarray(stats["stds"], dtype=np.float32).reshape(1, 1, 1, 4)

    processed = np.clip(images, lower, upper)          # 1) recorte de colas
    processed = (processed - means) / stds             # 2) estandarización
    processed = np.where(masks[..., None], 0.0, processed)  # 3) máscara → 0
    return np.ascontiguousarray(processed, dtype=np.float32)


def translate_zero_fill(image, shift_y, shift_x):
    """Traslación entera rellenando con ceros (sin interpolación)."""
    output = np.zeros_like(image)
    height, width = image.shape[:2]

    # Ventana de origen y destino según el signo del desplazamiento
    src_y0, src_y1 = max(0, -shift_y), min(height, height - shift_y)
    src_x0, src_x1 = max(0, -shift_x), min(width, width - shift_x)
    dst_y0, dst_x0 = max(0, shift_y), max(0, shift_x)
    dst_y1 = dst_y0 + (src_y1 - src_y0)
    dst_x1 = dst_x0 + (src_x1 - src_x0)

    output[dst_y0:dst_y1, dst_x0:dst_x1] = image[
        src_y0:src_y1, src_x0:src_x1
    ]
    return output


def augment_batch(images, seed, max_translation=MAX_TRANSLATION_PIXELS):
    """Rotación 0/90/180/270 + reflexiones + traslación entera (por imagen).

    Las transformaciones actúan sobre los ejes espaciales (0,1) de arreglos
    (H, W, 4): las 4 bandas se transforman JUNTAS y quedan alineadas.
    """
    rng = np.random.default_rng(seed)
    augmented = np.empty_like(images)

    for index, image in enumerate(images):
        transformed = np.rot90(image, k=int(rng.integers(0, 4)), axes=(0, 1))
        if rng.random() < 0.5:
            transformed = np.flip(transformed, axis=1)   # espejo horizontal
        if rng.random() < 0.5:
            transformed = np.flip(transformed, axis=0)   # espejo vertical
        shift_y = int(rng.integers(-max_translation, max_translation + 1))
        shift_x = int(rng.integers(-max_translation, max_translation + 1))
        augmented[index] = translate_zero_fill(transformed, shift_y, shift_x)

    return np.ascontiguousarray(augmented, dtype=np.float32)


### Generador de lotes: `H5LensDataset`

Entrega lotes desde H5, con shuffle y augmentation solo en train. A diferencia del Experimento 4, aqui siempre usamos el flujo normalizado del pipeline comun porque el fine-tuning se entrena batch a batch.


In [ ]:
class H5LensDataset(torch.utils.data.Dataset):
    def __init__(self, dataset_path, indices, stats, batch_size=None, shuffle=False, augment=False, seed=42):
        super().__init__()
        self.dataset_path = Path(dataset_path)
        self.indices = np.asarray(indices, dtype=np.int64).copy()
        self.stats = stats
        self.batch_size = int(BATCH_SIZE if batch_size is None else batch_size)
        self.shuffle = bool(shuffle)
        self.augment = bool(augment)
        self.seed = int(seed)
        self.epoch = 0
        self.order = self.indices.copy()
        self._set_epoch_order()

    def __len__(self):
        return math.ceil(len(self.order) / self.batch_size)

    def _set_epoch_order(self):
        self.order = self.indices.copy()
        if self.shuffle:
            rng = np.random.default_rng(self.seed + self.epoch)
            rng.shuffle(self.order)

    def __getitem__(self, batch_index):
        start = batch_index * self.batch_size
        stop = min(start + self.batch_size, len(self.order))
        batch_indices = self.order[start:stop]
        if len(batch_indices) == 0:
            raise IndexError(batch_index)
        batch = read_h5_batch(self.dataset_path, batch_indices)
        images = preprocess_images(batch["image"], batch["mask"], self.stats)
        if self.augment:
            batch_seed = self.seed + self.epoch * 1_000_003 + batch_index
            images = augment_batch(images, batch_seed)
        targets = batch["label"].astype(np.float32).reshape(-1, 1)
        return images, targets

    def on_epoch_end(self):
        self.epoch += 1
        self._set_epoch_order()

train_sequence = H5LensDataset(DATASET_PATH, indices_uso_train, preprocessing_stats, batch_size=BATCH_SIZE, shuffle=True, augment=True, seed=SEED)
val_sequence = H5LensDataset(DATASET_PATH, indices_uso_val, preprocessing_stats, batch_size=BATCH_SIZE, shuffle=False, augment=False, seed=SEED)
test_sequence = H5LensDataset(DATASET_PATH, indices_uso_test, preprocessing_stats, batch_size=BATCH_SIZE, shuffle=False, augment=False, seed=SEED)

train_x, train_y = train_sequence[0]
assert train_x.shape[1:] == (101, 101, 4)
assert train_y.shape == (len(train_x), 1)
assert np.isfinite(train_x).all()
print("Lotes train/validation/test:", len(train_sequence), len(val_sequence), len(test_sequence))
print("Primer lote:", train_x.shape, train_y.shape)


### Demostracion del augmentation

Verificacion visual de que las 4 bandas se transforman juntas.


In [ ]:
# Tomamos 4 imágenes preprocesadas de train y les aplicamos el augmentation
demo = read_h5_batch(DATASET_PATH, indices_uso_train[:4])
demo_proc = preprocess_images(demo["image"], demo["mask"], preprocessing_stats)
demo_aug = augment_batch(demo_proc, seed=SEED)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col in range(4):
    axes[0, col].imshow(rgb_for_display(demo_proc[col]), origin="lower")
    axes[0, col].set_title(f"original · clase {demo['label'][col]}")
    axes[0, col].axis("off")
    axes[1, col].imshow(rgb_for_display(demo_aug[col]), origin="lower")
    axes[1, col].set_title("aumentada")
    axes[1, col].axis("off")
fig.suptitle("Augmentation: rotación 90° + reflexión + traslación ≤3 px", fontsize=13)
fig.tight_layout()
plt.show()


## 4. AION-1: carga, congelamiento y descongelamiento parcial

**QUE hace:** carga `polymathic-ai/aion-base`, congela todo AION y descongela solo los ultimos bloques/capas detectables del encoder.

**POR QUE se hace:** adapta el modelo a lentes sin caer en fine-tuning completo, que queda para el Experimento 6.

**QUE se reutiliza de Renzo:** `AION.from_pretrained`, `CodecManager`, `LegacySurveyImage`, canales g,r,i y `NUM_ENCODER_TOKENS=600`.

**QUE cambia:** AION tiene parametros entrenables en los ultimos bloques; el codec permanece congelado.


In [ ]:
from aion import AION
from aion.codecs import CodecManager
from aion.modalities import LegacySurveyImage

CANALES_GRI = [1, 2, 3]
BANDAS_AION = ["DES-G", "DES-R", "DES-I"]
NUM_ENCODER_TOKENS = 600
EMBEDDING_DIM = 768

N_UNFROZEN_BLOCKS = int(os.environ.get("EXP5_N_UNFROZEN_BLOCKS", "1"))
HEAD_LR = float(os.environ.get("EXP5_HEAD_LR", "1e-3"))
BACKBONE_LR_MULT = float(os.environ.get("EXP5_BACKBONE_LR_MULT", "0.1"))
BACKBONE_LR = HEAD_LR * BACKBONE_LR_MULT
WEIGHT_DECAY = float(os.environ.get("EXP5_WEIGHT_DECAY", "1e-4"))
MAX_EPOCHS = int(os.environ.get("EXP5_MAX_EPOCHS", "10" if not PRUEBA_RAPIDA else "2"))
PATIENCE = int(os.environ.get("EXP5_PATIENCE", "3" if not PRUEBA_RAPIDA else "1"))
GRAD_CLIP_NORM = float(os.environ.get("EXP5_GRAD_CLIP_NORM", "1.0"))
WARMUP_RATIO = float(os.environ.get("EXP5_WARMUP_RATIO", "0.1"))
HEAD_HIDDEN_DIM = 256
DROPOUT = 0.3
USE_AMP = DEVICE.type == "cuda"

# Si la estructura exacta de AION cambia, completar esta lista con regex de parametros.
UNFREEZE_PATTERNS = []
ALLOW_FULL_FINETUNING = False
INSPECT_AION_STRUCTURE = True

def freeze_all_aion(modelo_aion):
    for p in modelo_aion.parameters():
        p.requires_grad = False

def _candidate_block_containers(modelo_aion):
    candidates = []
    terms = ("block", "blocks", "layer", "layers", "encoder", "transformer")
    for module_name, module in modelo_aion.named_modules():
        children = list(module.named_children())
        if len(children) < 2:
            continue
        child_names = [name for name, _ in children]
        numeric_children = sum(name.isdigit() for name in child_names)
        lower_name = module_name.lower()
        term_hits = sum(term in lower_name for term in terms)
        is_module_list = isinstance(module, nn.ModuleList)
        repeated_children = numeric_children >= max(2, len(children) // 2)
        if not (is_module_list or repeated_children or term_hits):
            continue
        if sum(p.numel() for p in module.parameters(recurse=True)) == 0:
            continue
        score = 10 * int(is_module_list) + 5 * int(repeated_children) + term_hits + min(len(children), 48) / 100
        if "encoder" in lower_name:
            score += 3
        if "decoder" in lower_name:
            score -= 3
        candidates.append({"name": module_name, "module": module, "children": children, "n_children": len(children), "score": score})
    candidates.sort(key=lambda x: (x["score"], x["n_children"]), reverse=True)
    return candidates

def inspect_aion_blocks(modelo_aion, max_candidates=12, max_groups=40):
    candidates = _candidate_block_containers(modelo_aion)
    print("Candidatos de contenedores de bloques/capas:")
    for i, cand in enumerate(candidates[:max_candidates], 1):
        preview = [name for name, _ in cand["children"][:6]]
        print(f" {i:02d}. name='{cand['name'] or '<root>'}' | children={cand['n_children']} | score={cand['score']:.2f} | hijos={preview}")
    if not candidates:
        print(" No se detectaron candidatos; usar UNFREEZE_PATTERNS.")

    grouped = {}
    for name, param in modelo_aion.named_parameters():
        prefix = ".".join(name.split(".")[:3])
        item = grouped.setdefault(prefix, {"params": 0, "tensors": 0, "examples": []})
        item["params"] += param.numel()
        item["tensors"] += 1
        if len(item["examples"]) < 2:
            item["examples"].append(name)
    print("\nGrupos de named_parameters:")
    for i, (prefix, item) in enumerate(grouped.items(), 1):
        if i > max_groups:
            print(f" ... {len(grouped) - max_groups} grupos mas omitidos")
            break
        print(f" {prefix:<55} tensors={item['tensors']:<4} params={item['params']:,} ejemplos={item['examples']}")
    return candidates

def _print_trainable_summary(modelo_aion):
    total = sum(p.numel() for p in modelo_aion.parameters())
    trainable = sum(p.numel() for p in modelo_aion.parameters() if p.requires_grad)
    percent = 100 * trainable / total if total else 0.0
    names = [name for name, p in modelo_aion.named_parameters() if p.requires_grad]
    print(f"Parametros totales AION:     {total:,}")
    print(f"Parametros entrenables AION: {trainable:,} ({percent:.4f}%)")
    for name in names:
        print(" -", name)
    return {"aion_total_params": int(total), "aion_trainable_params": int(trainable), "aion_trainable_percent": float(percent), "aion_trainable_names": names}

def unfreeze_last_aion_blocks(modelo_aion, n_blocks=2, patterns=None, allow_full=False):
    if allow_full:
        raise RuntimeError("No descongeles todo AION en el Experimento 5; eso corresponde al Experimento 6.")
    freeze_all_aion(modelo_aion)
    patterns = patterns or []
    if patterns:
        compiled = [re.compile(pattern) for pattern in patterns]
        for name, p in modelo_aion.named_parameters():
            if any(pattern.search(name) for pattern in compiled):
                p.requires_grad = True
        print("Descongelamiento por patrones:", patterns)
    else:
        candidates = _candidate_block_containers(modelo_aion)
        if not candidates:
            _print_trainable_summary(modelo_aion)
            raise RuntimeError("No se detectaron bloques automaticamente. Ejecuta inspect_aion_blocks y define UNFREEZE_PATTERNS.")
        chosen = candidates[0]
        selected = chosen["children"][-int(n_blocks):]
        print(f"Contenedor seleccionado: '{chosen['name'] or '<root>'}' con {chosen['n_children']} hijos")
        print("Bloques descongelados:", [name for name, _ in selected])
        for _, child_module in selected:
            for p in child_module.parameters(recurse=True):
                p.requires_grad = True
    summary = _print_trainable_summary(modelo_aion)
    if summary["aion_trainable_params"] == 0:
        raise RuntimeError("AION quedo sin parametros entrenables. Ajusta N_UNFROZEN_BLOCKS o UNFREEZE_PATTERNS.")
    return summary

modelo_aion = AION.from_pretrained("polymathic-ai/aion-base").to(DEVICE)
codec_manager = CodecManager(device=DEVICE)
freeze_all_aion(modelo_aion)
if INSPECT_AION_STRUCTURE:
    _ = inspect_aion_blocks(modelo_aion)
aion_param_summary = unfreeze_last_aion_blocks(modelo_aion, N_UNFROZEN_BLOCKS, UNFREEZE_PATTERNS, ALLOW_FULL_FINETUNING)


## 5. Modelo: AION parcial + cabeza MLP

**QUE hace:** define `AIONPartialFineTuner`.

**POR QUE se hace:** al cambiar los ultimos bloques de AION, los embeddings son dinamicos y deben recalcularse por batch.

**QUE se reutiliza de Renzo:** `LegacySurveyImage`, `codec_manager.encode`, `modelo_aion.encode` y mean pooling.

**QUE cambia:** no hay embeddings precalculados ni `StandardScaler`; la cabeza empieza con `LayerNorm`.


In [ ]:
def batch_to_aion_tensor(images, device=DEVICE):
    flujo = np.ascontiguousarray(images[..., CANALES_GRI].transpose(0, 3, 1, 2))
    tensor = torch.from_numpy(flujo).float().to(device, non_blocking=True)
    assert tensor.ndim == 4 and tensor.shape[1:] == (3, 101, 101)
    return tensor

class AIONPartialFineTuner(nn.Module):
    def __init__(self, modelo_aion, codec_manager, embedding_dim=EMBEDDING_DIM, hidden_dim=HEAD_HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.modelo_aion = modelo_aion
        self.codec_manager = codec_manager
        self.embedding_dim = int(embedding_dim)
        self.head = nn.Sequential(
            nn.LayerNorm(self.embedding_dim),
            nn.Linear(self.embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, flux_tensor):
        # El codec/tokenizer queda congelado. El encoder NO va dentro de no_grad.
        with torch.no_grad():
            image = LegacySurveyImage(flux=flux_tensor, bands=BANDAS_AION)
            tokens = self.codec_manager.encode(image)
        encoded = self.modelo_aion.encode(tokens, num_encoder_tokens=NUM_ENCODER_TOKENS)
        pooled = encoded.mean(dim=1)
        if pooled.shape[-1] != self.embedding_dim:
            raise RuntimeError(f"Embedding dim inesperada: {pooled.shape[-1]} != {self.embedding_dim}")
        return self.head(pooled).squeeze(-1)

def summarize_full_model(modelo):
    total = sum(p.numel() for p in modelo.parameters())
    trainable = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    percent = 100 * trainable / total if total else 0.0
    head_trainable = sum(p.numel() for p in modelo.head.parameters() if p.requires_grad)
    aion_trainable = sum(p.numel() for p in modelo.modelo_aion.parameters() if p.requires_grad)
    names = [name for name, p in modelo.named_parameters() if p.requires_grad]
    print(f"Parametros totales modelo:     {total:,}")
    print(f"Parametros entrenables modelo: {trainable:,} ({percent:.4f}%)")
    print(f" - AION entrenable:   {aion_trainable:,}")
    print(f" - cabeza entrenable: {head_trainable:,}")
    return {"total_params": int(total), "trainable_params": int(trainable), "trainable_percent": float(percent), "head_trainable_params": int(head_trainable), "aion_trainable_params": int(aion_trainable), "trainable_parameter_names": names}

modelo = AIONPartialFineTuner(modelo_aion, codec_manager).to(DEVICE)
param_summary = summarize_full_model(modelo)
aion_total = sum(p.numel() for p in modelo.modelo_aion.parameters())
aion_trainable = sum(p.numel() for p in modelo.modelo_aion.parameters() if p.requires_grad)
assert 0 < aion_trainable < aion_total, "AION debe estar parcialmente, no completamente, descongelado."
assert all(p.requires_grad for p in modelo.head.parameters()), "La cabeza debe ser entrenable."
print("Validacion OK: cabeza + ultimos bloques de AION son entrenables.")


## 6. Metricas, optimizador y scheduler

**QUE hace:** define metricas comparables, AdamW con dos grupos y scheduler coseno con warmup.

**POR QUE se hace:** la cabeza aprende rapido; el backbone preentrenado requiere LR menor.

**QUE se reutiliza de Renzo:** Youden, AUROC, AUPRC, Brier, log-loss, TPR0 y TPR10.

**QUE cambia:** optimizador con `BACKBONE_LR = HEAD_LR * 0.1`.


In [ ]:
def metrics_at_threshold(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions).ravel()
    return {"threshold": float(threshold), "accuracy": float(accuracy_score(y_true, predictions)), "precision": float(precision_score(y_true, predictions, zero_division=0)), "recall": float(recall_score(y_true, predictions, zero_division=0)), "f1": float(f1_score(y_true, predictions, zero_division=0)), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

def maximum_tpr_with_fp_limit(y_true, probabilities, max_fp):
    order = np.argsort(-probabilities)
    sorted_labels = np.asarray(y_true)[order]
    tp = np.cumsum(sorted_labels == 1)
    fp = np.cumsum(sorted_labels == 0)
    valid = fp <= max_fp
    best_tp = int(tp[valid].max()) if valid.any() else 0
    positives = int(np.sum(y_true == 1))
    return best_tp / positives if positives else 0.0

def umbral_youden(y_true, probabilities):
    fpr, tpr, thresholds = roc_curve(y_true, probabilities)
    finite = np.isfinite(thresholds)
    j = np.argmax((tpr - fpr)[finite])
    return float(thresholds[finite][j])

def metricas_completas(y_true, probabilities, threshold):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    assert np.all((0.0 <= probabilities) & (probabilities <= 1.0)), "Probabilidades fuera de [0,1]"
    clipped = np.clip(probabilities, 1e-7, 1 - 1e-7)
    m = metrics_at_threshold(y_true, probabilities, threshold)
    m.update({"auroc": float(roc_auc_score(y_true, probabilities)), "auprc": float(average_precision_score(y_true, probabilities)), "brier": float(brier_score_loss(y_true, probabilities)), "log_loss": float(log_loss(y_true, clipped)), "tpr_at_0_fp": float(maximum_tpr_with_fp_limit(y_true, probabilities, 0)), "tpr_below_10_fp": float(maximum_tpr_with_fp_limit(y_true, probabilities, 9))})
    return m

def build_optimizer(modelo):
    backbone_params = [p for p in modelo.modelo_aion.parameters() if p.requires_grad]
    head_params = [p for p in modelo.head.parameters() if p.requires_grad]
    if not backbone_params:
        raise RuntimeError("No hay parametros entrenables de AION.")
    return torch.optim.AdamW([
        {"params": backbone_params, "lr": BACKBONE_LR, "name": "backbone"},
        {"params": head_params, "lr": HEAD_LR, "name": "head"},
    ], weight_decay=WEIGHT_DECAY)

def build_cosine_warmup_scheduler(optimizer, total_steps, warmup_ratio=WARMUP_RATIO):
    warmup_steps = max(1, int(total_steps * warmup_ratio))
    total_steps = max(1, int(total_steps))
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step + 1) / float(warmup_steps)
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def autocast_context():
    if not USE_AMP:
        return nullcontext()
    try:
        return torch.amp.autocast("cuda")
    except AttributeError:
        return torch.cuda.amp.autocast()

def make_grad_scaler():
    if not USE_AMP:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)

def current_lrs(optimizer):
    return {group.get("name", f"group_{idx}"): float(group["lr"]) for idx, group in enumerate(optimizer.param_groups)}


## 7. Bucle de entrenamiento PyTorch

**QUE hace:** entrena con forward de AION por batch, BCEWithLogitsLoss, AMP si hay CUDA, gradient clipping, scheduler por step y early stopping.

**POR QUE se hace:** los embeddings cambian durante fine-tuning, asi que el entrenamiento debe ser end-to-end y auditable.

**QUE se reutiliza de Renzo:** early stopping, validacion por epoca y curvas/logs.

**QUE cambia:** el encoder entrenable de AION NO esta dentro de `torch.no_grad()`; solo validacion/test usan `torch.no_grad()`.


In [ ]:
BEST_MODEL_PATH = ARTIFACTS_DIR / "best_model.pt"
TRAINING_LOG_PATH = ARTIFACTS_DIR / "training_log.csv"
CONFIG_PATH = ARTIFACTS_DIR / "config.json"

CONFIG = {
    "experimento": "experimento5_aion_finetuning_parcial",
    "modelo_base": "polymathic-ai/aion-base",
    "seed": SEED,
    "dataset": SOURCE_DATASET_PATH.name,
    "batch_size": BATCH_SIZE,
    "n_train": int(len(indices_uso_train)),
    "n_val": int(len(indices_uso_val)),
    "n_test": int(len(indices_uso_test)),
    "prueba_rapida": bool(PRUEBA_RAPIDA),
    "n_unfrozen_blocks": int(N_UNFROZEN_BLOCKS),
    "unfreeze_patterns": UNFREEZE_PATTERNS,
    "head_lr": HEAD_LR,
    "backbone_lr": BACKBONE_LR,
    "backbone_lr_mult": BACKBONE_LR_MULT,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "warmup_ratio": WARMUP_RATIO,
    "bands_used": BANDAS_AION,
    "num_encoder_tokens": NUM_ENCODER_TOKENS,
    "use_amp": bool(USE_AMP),
    **param_summary,
}
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print("Configuracion guardada en:", CONFIG_PATH)

def trainable_aion_state_dict(modelo):
    trainable_names = {name for name, p in modelo.modelo_aion.named_parameters() if p.requires_grad}
    state = modelo.modelo_aion.state_dict()
    return {name: tensor.detach().cpu() for name, tensor in state.items() if name in trainable_names}

def save_partial_checkpoint(modelo, path, epoch, val_loss, val_auroc, config):
    checkpoint = {
        "experimento": "experimento5_aion_finetuning_parcial",
        "modelo_base": "polymathic-ai/aion-base",
        "epoch": int(epoch),
        "val_loss": float(val_loss),
        "val_auroc": float(val_auroc),
        "head_state_dict": {k: v.detach().cpu() for k, v in modelo.head.state_dict().items()},
        "aion_trainable_state_dict": trainable_aion_state_dict(modelo),
        "config": config,
    }
    torch.save(checkpoint, path)

def load_partial_checkpoint(modelo, path, device=DEVICE):
    try:
        checkpoint = torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location=device)
    modelo.head.load_state_dict(checkpoint["head_state_dict"])
    named_params = dict(modelo.modelo_aion.named_parameters())
    with torch.no_grad():
        for name, tensor in checkpoint["aion_trainable_state_dict"].items():
            if name not in named_params:
                raise KeyError(f"Parametro de AION no encontrado: {name}")
            named_params[name].copy_(tensor.to(device))
    return checkpoint

@torch.no_grad()
def evaluar_secuencia(modelo, sequence, criterio):
    modelo.eval()
    total_loss, total_n = 0.0, 0
    all_probabilities, all_labels = [], []
    for batch_index in range(len(sequence)):
        images, targets_np = sequence[batch_index]
        x = batch_to_aion_tensor(images, DEVICE)
        y = torch.from_numpy(targets_np.ravel()).float().to(DEVICE)
        with autocast_context():
            logits = modelo(x)
            loss = criterio(logits, y)
        if not torch.isfinite(logits).all() or not torch.isfinite(loss):
            raise FloatingPointError("NaN/Inf en evaluacion")
        all_probabilities.append(torch.sigmoid(logits.float()).cpu().numpy())
        all_labels.append(targets_np.ravel().astype(np.int8))
        total_loss += float(loss.item()) * len(y)
        total_n += len(y)
    y_true = np.concatenate(all_labels)
    probabilities = np.concatenate(all_probabilities)
    assert np.all((0.0 <= probabilities) & (probabilities <= 1.0))
    return total_loss / total_n, y_true, probabilities

def entrenar_una_epoca(modelo, sequence, criterio, optimizer, scheduler, scaler, epoch):
    modelo.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss, n_seen = 0.0, 0
    checked_backbone_grad = False
    for batch_index in range(len(sequence)):
        images, targets_np = sequence[batch_index]
        x = batch_to_aion_tensor(images, DEVICE)
        y = torch.from_numpy(targets_np.ravel()).float().to(DEVICE)
        with autocast_context():
            logits = modelo(x)
            loss = criterio(logits, y)
        if not torch.isfinite(logits).all() or not torch.isfinite(loss):
            raise FloatingPointError(f"NaN/Inf epoch={epoch}, batch={batch_index}")

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), GRAD_CLIP_NORM)
            if not checked_backbone_grad:
                assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in modelo.modelo_aion.parameters() if p.requires_grad), "No hay gradientes en AION descongelado"
                checked_backbone_grad = True
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), GRAD_CLIP_NORM)
            if not checked_backbone_grad:
                assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in modelo.modelo_aion.parameters() if p.requires_grad), "No hay gradientes en AION descongelado"
                checked_backbone_grad = True
            optimizer.step()

        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        running_loss += float(loss.item()) * len(y)
        n_seen += len(y)
        if (batch_index + 1) % 25 == 0 or batch_index == len(sequence) - 1:
            print(f"  epoch {epoch:02d} | batch {batch_index + 1:04d}/{len(sequence):04d} | loss={loss.item():.4f}")
    return running_loss / n_seen

def fit_aion_partial(modelo, train_sequence, val_sequence, checkpoint_path=BEST_MODEL_PATH):
    criterio = nn.BCEWithLogitsLoss()
    optimizer = build_optimizer(modelo)
    scheduler = build_cosine_warmup_scheduler(optimizer, max(1, len(train_sequence) * MAX_EPOCHS))
    scaler = make_grad_scaler()
    best_val_auroc, best_val_loss, best_epoch = -np.inf, np.inf, 0
    epochs_without_improvement = 0
    history_rows = []
    start = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        print(f"\n===== Epoch {epoch}/{MAX_EPOCHS} =====")
        train_loss = entrenar_una_epoca(modelo, train_sequence, criterio, optimizer, scheduler, scaler, epoch)
        val_loss, y_val_epoch, prob_val_epoch = evaluar_secuencia(modelo, val_sequence, criterio)
        val_auroc = roc_auc_score(y_val_epoch, prob_val_epoch)
        val_accuracy = accuracy_score(y_val_epoch, (prob_val_epoch >= 0.5).astype(np.int8))
        lrs = current_lrs(optimizer)
        row = {"epoch": epoch, "train_loss": float(train_loss), "val_loss": float(val_loss), "val_auroc": float(val_auroc), "val_accuracy": float(val_accuracy), "lr_head": float(lrs.get("head", np.nan)), "lr_backbone": float(lrs.get("backbone", np.nan))}
        history_rows.append(row)
        pd.DataFrame(history_rows).to_csv(TRAINING_LOG_PATH, index=False)
        print(f"epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_auroc={val_auroc:.4f} | val_acc@0.5={val_accuracy:.4f}")

        improved = val_auroc > best_val_auroc + 1e-6 or (abs(val_auroc - best_val_auroc) <= 1e-6 and val_loss < best_val_loss)
        if improved:
            best_val_auroc, best_val_loss, best_epoch = float(val_auroc), float(val_loss), int(epoch)
            epochs_without_improvement = 0
            save_partial_checkpoint(modelo, checkpoint_path, epoch, val_loss, val_auroc, CONFIG)
            print("  checkpoint actualizado:", checkpoint_path)
        else:
            epochs_without_improvement += 1

        train_sequence.on_epoch_end()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping en epoch {epoch}; mejor epoch={best_epoch}.")
            break

    elapsed_minutes = (time.time() - start) / 60
    checkpoint = load_partial_checkpoint(modelo, checkpoint_path, DEVICE)
    return {"best_epoch": best_epoch, "best_val_loss": best_val_loss, "best_val_auroc": best_val_auroc, "tiempo_entrenamiento_minutos": float(elapsed_minutes), "history": history_rows, "checkpoint": checkpoint}

fit_info = fit_aion_partial(modelo, train_sequence, val_sequence)
print("Mejor epoch:", fit_info["best_epoch"])
print("Mejor val_loss:", fit_info["best_val_loss"])
print("Mejor val_auroc:", fit_info["best_val_auroc"])
print("Tiempo entrenamiento (min):", fit_info["tiempo_entrenamiento_minutos"])


### Curvas de entrenamiento

Guarda `training_curves.png` y muestra `training_log.csv` con una fila por epoca.


In [ ]:
history_df = pd.DataFrame(fit_info["history"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="validation")
axes[0].set(xlabel="Epoca", ylabel="BCEWithLogitsLoss", title="Perdida")
axes[0].legend(); axes[0].grid(alpha=0.25)
axes[1].plot(history_df["epoch"], history_df["val_auroc"], marker="o", color="tab:green")
axes[1].set(xlabel="Epoca", ylabel="AUROC validacion", title="Checkpoint por AUROC")
axes[1].grid(alpha=0.25)
fig.suptitle("AION fine-tuning parcial: curvas de entrenamiento", fontsize=13)
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "training_curves.png", dpi=180, bbox_inches="tight")
plt.show()
history_df


## 8. Cross-validation opcional

**QUE hace:** deja preparada una CV estratificada liviana con `RUN_CV=True`.

**POR QUE se hace:** el reporte pide CV en general, pero en fine-tuning parcial repetir entrenamientos de AION es costoso.

**QUE se reutiliza de Renzo:** CV sobre train+val y test excluido.

**QUE cambia:** por defecto `RUN_CV=False`; se guarda `cv_results.json` indicando si se ejecuto.


In [ ]:
RUN_CV = False
CV_N_SPLITS = 5
CV_MAX_EPOCHS = 3
CV_SUBSET_SIZE = 1_000 if not PRUEBA_RAPIDA else 120

if RUN_CV:
    print("CV activada: puede tardar bastante.")
    cv_indices_all = np.concatenate([indices_uso_train, indices_uso_val])
    if CV_SUBSET_SIZE is not None and CV_SUBSET_SIZE < len(cv_indices_all):
        cv_indices_all = submuestra_estratificada(cv_indices_all, labels, CV_SUBSET_SIZE, SEED)
    cv_labels_all = labels[cv_indices_all]
    skf = StratifiedKFold(n_splits=CV_N_SPLITS, shuffle=True, random_state=SEED)
    cv_rows = []
    original_max_epochs = MAX_EPOCHS

    for fold, (train_pos, eval_pos) in enumerate(skf.split(cv_indices_all, cv_labels_all), 1):
        print(f"\nFold {fold}/{CV_N_SPLITS}")
        fold_train_indices = cv_indices_all[train_pos]
        fold_eval_indices = cv_indices_all[eval_pos]
        fold_train_sequence = H5LensDataset(DATASET_PATH, fold_train_indices, preprocessing_stats, batch_size=BATCH_SIZE, shuffle=True, augment=True, seed=SEED + fold)
        fold_eval_sequence = H5LensDataset(DATASET_PATH, fold_eval_indices, preprocessing_stats, batch_size=BATCH_SIZE, shuffle=False, augment=False, seed=SEED + fold)

        fold_aion = AION.from_pretrained("polymathic-ai/aion-base").to(DEVICE)
        freeze_all_aion(fold_aion)
        _ = unfreeze_last_aion_blocks(fold_aion, N_UNFROZEN_BLOCKS, UNFREEZE_PATTERNS, ALLOW_FULL_FINETUNING)
        fold_model = AIONPartialFineTuner(fold_aion, CodecManager(device=DEVICE)).to(DEVICE)
        MAX_EPOCHS = CV_MAX_EPOCHS
        fold_fit = fit_aion_partial(fold_model, fold_train_sequence, fold_eval_sequence, ARTIFACTS_DIR / f"best_model_cv_fold_{fold}.pt")
        MAX_EPOCHS = original_max_epochs

        criterion = nn.BCEWithLogitsLoss()
        _, y_eval, prob_eval = evaluar_secuencia(fold_model, fold_eval_sequence, criterion)
        threshold_fold = umbral_youden(y_eval, prob_eval)
        cv_rows.append({"fold": fold, **metricas_completas(y_eval, prob_eval, threshold_fold), "best_epoch": fold_fit["best_epoch"]})
        del fold_model, fold_aion
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    cv_df = pd.DataFrame(cv_rows)
    cv_summary = {"run_cv": True, "n_splits": CV_N_SPLITS, "cv_subset_size": int(len(cv_indices_all)), "folds": cv_rows, "summary": cv_df.drop(columns=["fold"]).agg(["mean", "std"]).to_dict()}
    (ARTIFACTS_DIR / "cv_results.json").write_text(json.dumps(cv_summary, indent=2), encoding="utf-8")
    display(cv_df)
else:
    cv_summary = {"run_cv": False, "reason": "Fine-tuning parcial es costoso; se prioriza train/val/test."}
    (ARTIFACTS_DIR / "cv_results.json").write_text(json.dumps(cv_summary, indent=2), encoding="utf-8")
    print("RUN_CV=False: cv_results.json guardado.")


## 9. Evaluacion final en validation/test

**QUE hace:** carga el mejor checkpoint, elige el umbral con validation y evalua test una sola vez.

**POR QUE se hace:** evita usar test para hiperparametros o threshold.

**QUE se reutiliza de Renzo:** Youden, metricas completas, bootstrap, ROC, PR y matriz de confusion.

**QUE cambia:** las probabilidades salen del AION parcialmente fine-tuned, no de embeddings fijos.


In [ ]:
criterion = nn.BCEWithLogitsLoss()
_ = load_partial_checkpoint(modelo, BEST_MODEL_PATH, DEVICE)

val_loss_final, y_val_final, prob_val_final = evaluar_secuencia(modelo, val_sequence, criterion)
selected_threshold = umbral_youden(y_val_final, prob_val_final)
print(f"Umbral seleccionado SOLO con validacion: {selected_threshold:.6f}")
print(f"Val loss final: {val_loss_final:.4f} | Val AUROC final: {roc_auc_score(y_val_final, prob_val_final):.4f}")

test_loss, y_test_final, prob_test_final = evaluar_secuencia(modelo, test_sequence, criterion)
assert np.all((0.0 <= prob_test_final) & (prob_test_final <= 1.0))
test_metrics = metricas_completas(y_test_final, prob_test_final, selected_threshold)
pd.Series(test_metrics, name="valor").to_frame()


In [ ]:
fpr, tpr, _ = roc_curve(y_test_final, prob_test_final)
precision_curve, recall_curve, _ = precision_recall_curve(y_test_final, prob_test_final)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f"AUROC = {test_metrics['auroc']:.4f}")
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="Curva ROC - AION fine-tuning parcial")
ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "roc_curve.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall_curve, precision_curve, label=f"AUPRC = {test_metrics['auprc']:.4f}")
ax.axhline(y_test_final.mean(), linestyle="--", color="gray", label="base")
ax.set(xlabel="Recall", ylabel="Precision", title="Curva Precision-Recall - AION fine-tuning parcial")
ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "pr_curve.png", dpi=180, bbox_inches="tight")
plt.show()

predicciones_test = (prob_test_final >= selected_threshold).astype(np.int8)
matriz = confusion_matrix(y_test_final, predicciones_test)
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(matriz, cmap="Blues")
for (fila, columna), valor in np.ndenumerate(matriz):
    ax.text(columna, fila, f"{valor}", ha="center", va="center", color="black" if valor < matriz.max() / 2 else "white", fontsize=14)
ax.set_xticks([0, 1], ["no lente", "lente"])
ax.set_yticks([0, 1], ["no lente", "lente"])
ax.set(xlabel="Prediccion", ylabel="Clase real", title=f"Matriz de confusion (umbral={selected_threshold:.3f})")
fig.colorbar(im, shrink=0.8)
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
BOOTSTRAP_REPETITIONS = 200 if PRUEBA_RAPIDA else 1_000
rng = np.random.default_rng(SEED)
positive_positions = np.flatnonzero(y_test_final == 1)
negative_positions = np.flatnonzero(y_test_final == 0)
bootstrap_rows = []

for _ in range(BOOTSTRAP_REPETITIONS):
    sampled = np.concatenate([
        rng.choice(positive_positions, size=len(positive_positions), replace=True),
        rng.choice(negative_positions, size=len(negative_positions), replace=True),
    ])
    sampled_y = y_test_final[sampled]
    sampled_p = prob_test_final[sampled]
    sampled_pred = (sampled_p >= selected_threshold).astype(np.int8)
    bootstrap_rows.append({
        "auroc": roc_auc_score(sampled_y, sampled_p),
        "auprc": average_precision_score(sampled_y, sampled_p),
        "accuracy": accuracy_score(sampled_y, sampled_pred),
        "recall": recall_score(sampled_y, sampled_pred, zero_division=0),
        "precision": precision_score(sampled_y, sampled_pred, zero_division=0),
        "f1": f1_score(sampled_y, sampled_pred, zero_division=0),
    })

bootstrap = pd.DataFrame(bootstrap_rows)
confidence_intervals = bootstrap.quantile([0.025, 0.5, 0.975]).T
confidence_intervals.columns = ["p2.5", "mediana", "p97.5"]
confidence_intervals


## 10. Guardado de resultados reproducibles

**QUE hace:** guarda metricas, config, predicciones, checkpoint, figuras y explicacion para video en `artifacts_experimento5/`.

**POR QUE se hace:** el resultado debe ser auditable y comparable.

**QUE se reutiliza de Renzo:** formato de `test_predictions.csv` y `metrics.json`.

**QUE cambia:** se agregan metadatos de fine-tuning parcial: bloques, parametros, LR y tiempo.


In [ ]:
def read_optional_h5_fields(dataset_path, indices, field_names):
    indices = np.asarray(indices, dtype=np.int64)
    order = np.argsort(indices)
    sorted_indices = indices[order]
    restore_order = np.argsort(order)
    result = {}
    with h5py.File(dataset_path, "r") as h5:
        for field in field_names:
            if field not in h5:
                continue
            values = h5[field][sorted_indices][restore_order]
            if getattr(values, "dtype", None) is not None and values.dtype.kind == "S":
                values = np.char.decode(values, "utf-8")
            result[field] = values
    return result

optional_test_metadata = read_optional_h5_fields(DATASET_PATH, indices_uso_test, ["theta_E", "lensed_snr_r", "negative_type"])
predictions_table = pd.DataFrame({"sample_index": indices_uso_test, "label": y_test_final, "probability": prob_test_final, "prediction": predicciones_test})
for field, values in optional_test_metadata.items():
    predictions_table[field] = values
predictions_table.to_csv(ARTIFACTS_DIR / "test_predictions.csv", index=False)

results = {
    "dataset": str(DATASET_PATH),
    "dataset_name": SOURCE_DATASET_PATH.name,
    "n_train": int(len(indices_uso_train)),
    "n_val": int(len(indices_uso_val)),
    "n_test": int(len(indices_uso_test)),
    "selected_threshold": float(selected_threshold),
    "test_loss": float(test_loss),
    "test_metrics": test_metrics,
    "bootstrap_95_percent": confidence_intervals.to_dict(orient="index"),
    "experimento": "experimento5_aion_finetuning_parcial",
    "modelo_base": "polymathic-ai/aion-base",
    "n_unfrozen_blocks": int(N_UNFROZEN_BLOCKS),
    "unfreeze_patterns": UNFREEZE_PATTERNS,
    "trainable_params": int(param_summary["trainable_params"]),
    "total_params": int(param_summary["total_params"]),
    "trainable_percent": float(param_summary["trainable_percent"]),
    "aion_trainable_params": int(param_summary["aion_trainable_params"]),
    "head_trainable_params": int(param_summary["head_trainable_params"]),
    "head_lr": float(HEAD_LR),
    "backbone_lr": float(BACKBONE_LR),
    "backbone_lr_mult": float(BACKBONE_LR_MULT),
    "batch_size": int(BATCH_SIZE),
    "max_epochs": int(MAX_EPOCHS),
    "best_epoch": int(fit_info["best_epoch"]),
    "best_val_loss": float(fit_info["best_val_loss"]),
    "best_val_auroc": float(fit_info["best_val_auroc"]),
    "bands_used": BANDAS_AION,
    "num_encoder_tokens": int(NUM_ENCODER_TOKENS),
    "prueba_rapida": bool(PRUEBA_RAPIDA),
    "tiempo_entrenamiento_minutos": float(fit_info["tiempo_entrenamiento_minutos"]),
}
(ARTIFACTS_DIR / "metrics.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
CONFIG.update({"best_epoch": results["best_epoch"], "best_val_loss": results["best_val_loss"], "best_val_auroc": results["best_val_auroc"], "tiempo_entrenamiento_minutos": results["tiempo_entrenamiento_minutos"]})
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

print("Resultados guardados en:", ARTIFACTS_DIR)
for name in ["metrics.json", "config.json", "training_log.csv", "test_predictions.csv", "training_curves.png", "roc_curve.png", "pr_curve.png", "confusion_matrix.png", "best_model.pt", "cv_results.json"]:
    path = ARTIFACTS_DIR / name
    print(f" - {name}:", "OK" if path.exists() else "pendiente")
if PRUEBA_RAPIDA:
    print("ATENCION: PRUEBA_RAPIDA=True; no son metricas finales.")


In [ ]:
explain_text = f"""# Experimento 5 - AION fine-tuning parcial

## Que se reutilizo del Experimento 4
- Secciones 1-3: semillas, lectura HDF5, split oficial, normalizacion train-only, mascara a cero y augmentation solo en train.
- Integracion AION: `LegacySurveyImage`, `CodecManager`, bandas `DES-G`, `DES-R`, `DES-I`, canales g,r,i y `NUM_ENCODER_TOKENS = 600`.
- Metricas del piloto/Renzo: accuracy, precision, recall, F1, AUROC, AUPRC, Brier, log-loss, TPR0, TPR10, matriz de confusion y bootstrap.

## Que cambia en el Experimento 5
- El Experimento 4 entrenaba una MLP sobre embeddings fijos.
- Aqui se descongelan solo los ultimos `{N_UNFROZEN_BLOCKS}` bloques detectados del encoder y se entrena junto con una cabeza MLP.

## Por que solo ultimas capas
Las primeras capas suelen capturar patrones generales; las ultimas son mas adaptables a la tarea. Asi reducimos sobreajuste y costo frente a fine-tuning completo.

## Por que LR diferenciado
La cabeza parte de cero y usa `HEAD_LR = {HEAD_LR}`. El backbone preentrenado usa `BACKBONE_LR = {BACKBONE_LR}`, diez veces menor, para no destruir representaciones astronomicas utiles.

## Por que no embeddings precalculados
Durante fine-tuning parcial los ultimos bloques de AION cambian despues de cada update. Por lo tanto los embeddings tambien cambian y deben recalcularse por batch.

## Como se evitan fugas de datos
Percentiles/media/std solo con train; augmentation solo en train; checkpoint y umbral solo con validation; test una sola vez al final.

## Como se calculan TPR0 y TPR10
Se ordena test por probabilidad descendente. TPR0 usa 0 falsos positivos. TPR10 usa maximo 9 falsos positivos, equivalente a menos de 10 FP.

## Como comparar contra AION congelado
Comparar accuracy, AUROC, AUPRC, F1, TPR0 y TPR10 junto con parametros entrenables y costo. Si la mejora es pequena, no se fuerza una conclusion: el resultado puede indicar que AION congelado ya captura casi toda la senal util.
"""
(ARTIFACTS_DIR / "explain_for_video.md").write_text(explain_text, encoding="utf-8")
print("Guardado:", ARTIFACTS_DIR / "explain_for_video.md")
print(explain_text)


## 11. Comparacion contra Experimento 4

**QUE hace:** carga `artifacts_experimento4/metrics.json` si existe; si no, usa la referencia temporal observada en el notebook de Renzo.

**POR QUE se hace:** responde si el fine-tuning parcial justifica su costo extra.

**QUE se reutiliza de Renzo:** metricas de test.

**QUE cambia:** tabla con parametros entrenables y costo aproximado.


In [ ]:
def _metric_from_results(obj, metric_name, default=np.nan):
    if not obj:
        return default
    test_metrics_obj = obj.get("test_metrics", {})
    return float(test_metrics_obj.get(metric_name, obj.get(metric_name, default)))

exp4_metrics_path = PROJECT_DIR / "artifacts_experimento4" / "metrics.json"
if exp4_metrics_path.exists():
    exp4_results = json.loads(exp4_metrics_path.read_text(encoding="utf-8"))
    exp4_source = str(exp4_metrics_path)
    exp4_trainable = exp4_results.get("trainable_params", 197_121)
else:
    exp4_results = None
    exp4_source = "referencia temporal: outputs observados en experimento4_aion_congelado_mlp.ipynb"
    exp4_trainable = 197_121

exp4_row = {
    "Modelo": "AION congelado + MLP (Exp. 4)",
    "Accuracy": _metric_from_results(exp4_results, "accuracy", 0.735),
    "AUROC": _metric_from_results(exp4_results, "auroc", 0.810),
    "AUPRC": _metric_from_results(exp4_results, "auprc", 0.833),
    "F1": _metric_from_results(exp4_results, "f1", 0.712),
    "TPR0": _metric_from_results(exp4_results, "tpr_at_0_fp", 0.118),
    "TPR10": _metric_from_results(exp4_results, "tpr_below_10_fp", 0.278),
    "parametros entrenables": exp4_trainable,
    "costo aproximado": "bajo: embeddings una vez + MLP",
    "fuente": exp4_source,
}
exp5_row = {
    "Modelo": "AION fine-tuning parcial (Exp. 5)",
    "Accuracy": test_metrics["accuracy"],
    "AUROC": test_metrics["auroc"],
    "AUPRC": test_metrics["auprc"],
    "F1": test_metrics["f1"],
    "TPR0": test_metrics["tpr_at_0_fp"],
    "TPR10": test_metrics["tpr_below_10_fp"],
    "parametros entrenables": int(param_summary["trainable_params"]),
    "costo aproximado": f"alto: {fit_info['best_epoch']} epocas con backward parcial de AION",
    "fuente": "artifacts_experimento5/metrics.json",
}
tabla_comparativa = pd.DataFrame([exp4_row, exp5_row])
tabla_comparativa.to_csv(ARTIFACTS_DIR / "comparison_experimento4_vs_5.csv", index=False)
print("Fuente Experimento 4:", exp4_source)
print("Pregunta central: mejora suficiente para justificar costo computacional?")
display(tabla_comparativa.round(4))
if not exp4_metrics_path.exists():
    print("Nota: no se encontro metrics.json del Experimento 4; la fila de Exp. 4 es referencia temporal, no fuente definitiva.")


## 12. Guion breve para el video

1. Recordar el pipeline comun: mismo H5, mismo split, normalizacion solo con train y augmentation solo en train.
2. Explicar la diferencia clave contra Renzo: no usamos embeddings fijos porque AION cambia durante fine-tuning.
3. Mostrar `freeze_all_aion`, `inspect_aion_blocks` y `unfreeze_last_aion_blocks`: se descongela solo el final del encoder.
4. Explicar LR diferenciado: cabeza aprende rapido, backbone se ajusta con LR 10 veces menor.
5. Mostrar metricas y tabla contra Experimento 4: si mejora, discutir si justifica costo; si no mejora, decirlo honestamente.
